# Lesson 02 - Microsoft Agent Frameworkin tutkiminen

**Microsoft Agent Framework (MAF)** on yhtenäinen kehys tekoälyagenttien rakentamiseen. Se tarjoaa selkeän, koostettavan arkkitehtuurin, jossa on neljä keskeistä rakennuspalikkaa:

- **Client** – yhdistää tekoälymallin päätepisteeseen ja hoitaa viestinnän
- **Agent** – käärii clientin ohjeilla ja työkalumääritelmillä
- **Tools** – laajentavat agentin kykyjä mallin kutsumilla mukautetuilla toiminnoilla
- **Session** – ylläpitää keskusteluhistoriaa monikertaisille vuorovaikutuksille

Tässä oppitunnissa rakennamme **matkavarauksen agentin**, joka tarkistaa kohteen saatavuuden näiden käsitteiden avulla.


## Asennus


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Agenttikehyksen arkkitehtuurin ymmärtäminen

Microsoftin Agenttikehys noudattaa kerroksittaista arkkitehtuuria:

```
Client  →  Agent  →  Tools
                  →  Session
```

1. **Asiakas** – `AzureAIProjectAgentProvider` yhdistää Azure OpenAI -käyttöönottoon. Se hoitaa todennuksen, pyyntöjen muotoilun ja vastausten jäsentämisen.
2. **Agentti** – Luodaan asiakkaasta metodilla `provider.create_agent()`, agentti yhdistää mallin käytön ohjeisiin (järjestelmän kehotus) ja työkaluihin.
3. **Työkalut** – Python-funktioita, jotka on koristeltu `@tool`-koristeella, ja joita agentti voi kutsua suorittaakseen toimintoja tai hakeakseen tietoja.
4. **Istunto** – `AgentSession`-olio (luodaan metodilla `agent.create_session()`), joka tallentaa keskusteluhistorian mahdollistaen monikierroksisen vuoropuhelun, jossa agentti muistaa aiemman kontekstin.

Rakennetaan jokainen kerros vaihe vaiheelta.


In [ ]:
# Create the client – this is the connection to the AI model
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Työkalujen lisääminen @tool-koristelijalla

Työkalut antavat agenteille mahdollisuuden tehdä toimintoja tekstin generoinnin lisäksi. `@tool`-koristelija muuttaa tavallisen Python-funktion sellaiseksi, jota agentti voi kutsua.

Tärkeimmät kohdat:
- Käytä `Annotated[type, "kuvaus"]`, jotta malli ymmärtää jokaisen parametrin.
- Menetelmäkuvaus (docstring) toimii työkalun kuvauksena, jonka malli näkee.
- `approval_mode="never_require"` tarkoittaa, että työkalu suoritetaan automaattisesti ilman käyttäjän vahvistusta.


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## Agentin luominen työkaluilla

Nyt yhdistämme asiakkaan, ohjeet ja työkalut agentiksi. `instructions` toimivat järjestelmän kehotteena — ne määrittelevät agentin persoonan ja käyttäytymisen.


In [ ]:
agent = await provider.create_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## Monikierroksiset keskustelut istuntojen avulla

`AgentSession` (luotu metodilla `agent.create_session()`) seuraa kaikkia keskustelun viestejä. Kun välitämme saman istunnon jokaiselle `agent.run()` -kutsulle, agentilla on pääsy koko keskusteluhistoriaan ja se voi viitata aiempiin viesteihin.

Välitämme `tools=[check_destination_availability]`, jotta agentti voi kutsua saatavuuden tarkistajaa jokaisen vuoron aikana.


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## Yhteenveto

Tässä oppitunnissa tutustuit Microsoft Agent Frameworkin neljään pilariin:

| Käsite | Mitä opit |
|---------|------------------|
| **Asiakas** | `AzureAIProjectAgentProvider` yhdistyy Azure OpenAI:hin tunnistetiedolla |
| **Agentti** | `provider.create_agent()` yhdistää malliyhteyden ohjeistuksiin ja nimeen |
| **Työkalut** | `@tool`-koristetta käytetään Python-funktioiden esille tuomiseen agentin kutsuttaviksi |
| **Istunto** | `agent.create_session()` ylläpitää keskusteluhistoriaa useiden kierrosten ajan |

Nämä rakennuspalikat muodostavat yhdessä agentteja, jotka pystyvät käymään luonnollisia keskusteluja, kutsumaan ulkoisia funktioita ja ylläpitämään kontekstia — perusta kehittyneemmille agenttipatternille myöhemmissä oppitunneissa.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastuuvapauslauseke**:
Tämä asiakirja on käännetty tekoälypohjaisella käännöspalvelulla [Co-op Translator](https://github.com/Azure/co-op-translator). Pyrimme tarkkuuteen, mutta huomioithan, että automaattiset käännökset saattavat sisältää virheitä tai epätarkkuuksia. Alkuperäistä asiakirjaa sen alkuperäisellä kielellä tulee pitää virallisena lähteenä. Tärkeissä asioissa suositellaan ammattimaista ihmiskäännöstä. Emme ole vastuussa tämän käännöksen käytöstä aiheutuvista väärinkäsityksistä tai tulkinnoista.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
